In [1]:
!git clone https://github.com/rasbt/reasoning-from-scratch.git
%cd reasoning-from-scratch
!pip install -r requirements.txt
!pip install sympy antlr4-python3-runtime==4.11.1

fatal: destination path 'reasoning-from-scratch' already exists and is not an empty directory.
/content/reasoning-from-scratch


# Knowledge Distillation Training

## What This Code Does
Trains a small "student" model (Qwen3-0.6B) to mimic a larger "teacher" model's reasoning style and answer format on math problems.

## The 3-Step Process

### 1. **Load Teacher's Examples**
- Downloads pre-generated solutions from a teacher model (GPT-4/Claude)
- Teacher outputs include `<think>` tags with reasoning + final answer
- Example: `<think>2+2=4</think>\boxed{4}`

### 2. **Train Student to Clone Teacher**
```
For each (problem, teacher_solution) pair:
  1. Concatenate: [PROMPT] + [TEACHER_RESPONSE]
  2. Mask the prompt (set to -100) so model only learns the response
  3. Compute loss: How well can student predict teacher's next token?
  4. Backpropagate & update weights
```

**Key Innovation**: Only train on teacher's response, not the problem itself → model learns *how to answer*, not memorization.

### 3. **Evaluate on Unseen Problems**
- Test on MATH-500 benchmark
- Check two things:
  1. **Accuracy**: Did it get the right answer?
  2. **Style Transfer**: Does it use `<think>` tags like teacher?

## Critical Implementation Details

| Component | Purpose |
|-----------|---------|
| **Sequence Truncation** | Limit to 1024 tokens to prevent GPU OOM on Colab T4 |
| **bfloat16 Precision** | Use 16-bit floats instead of 32-bit → 50% memory savings |
| **Greedy Decoding** | Always pick most likely token (simplest generation) |
| **KV Cache** | Store attention computations → faster generation |
| **Gradient Clipping** | Prevent exploding gradients (cap at 1.0) |

## Loss Function (The Core)
```python
Cross-Entropy Loss on shifted sequences:
  shift_logits = model_predictions[:-1]  # Predict tokens 0→n-1
  shift_labels = true_tokens[1:]         # Target tokens 1→n
  
Masking: prompt_tokens = -100 (ignored by loss)
Result: Model only learns to generate teacher's response
```

## Success Criteria
- **Pre-training**: Model knows language, but can't solve math
- **Post-distillation**: Model mimics teacher's reasoning format AND gets correct answers

## Why This Works
Teacher model is too expensive to run at scale → distill its knowledge into a 100x smaller model that's cheap to deploy but retains ~80% of teacher's capabilities.

In [4]:
import torch
import requests
import json
from pathlib import Path

# Base Imports & Tools
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Model, Qwen3Tokenizer, QWEN_CONFIG_06_B
from reasoning_from_scratch.ch03 import render_prompt

# ═══════════════════════════════════════════════════════════════════════════════
# KNOWLEDGE DISTILLATION TRAINING FOR LLMs
# ═══════════════════════════════════════════════════════════════════════════════
# This script implements Supervised Fine-Tuning (SFT) to distill knowledge from
# a larger "teacher" model into a smaller "student" model. The student learns to
# mimic the teacher's reasoning patterns and answer format.
# ═══════════════════════════════════════════════════════════════════════════════

# ---------------------------------------------------------
# STEP 1: Load Teacher's Outputs (Training Data)
# ---------------------------------------------------------
def load_distillation_sample():
    """
    Downloads pre-generated outputs from a teacher model (e.g., GPT-4, Claude).

    The teacher model has already solved problems and included its reasoning
    in <think> tags. Our goal is to teach the smaller student model to replicate
    this behavior.

    Returns:
        list: Training examples with format:
              [{
                  "problem": "What is 2+2?",
                  "message_thinking": "I need to add 2 and 2...",
                  "message_content": "\\boxed{4}"
              }, ...]
    """
    url = "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/main/ch08/02_generate_distillation_data/sample_ollama_outputs.json"
    print("📥 Fetching Teacher Distillation Data from GitHub...")

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        text = r.text.strip()

        # Handle both JSON array and JSONL (newline-delimited) formats
        if text.startswith('['):
            return json.loads(text)
        else:
            return [json.loads(line) for line in text.split('\n')]

    except Exception as e:
        print(f"⚠️  Download failed: {e}")
        print("📦 Using fallback mock data for demonstration...")
        return [{
            "problem": "What is 2+2?",
            "message_thinking": "To find the sum of 2 and 2, I will add them. 2 + 2 = 4.",
            "message_content": "\\boxed{4}"
        }]

# ---------------------------------------------------------
# STEP 2: Compute Training Loss (How Well Student Mimics Teacher)
# ---------------------------------------------------------
def compute_sft_loss(model, tokenizer, problem, teacher_text, device, max_seq_len=1024):
    """
    Calculates how well the student model can predict the teacher's response.

    This uses Causal Language Modeling (CLM) loss, which trains the model to
    predict the next token given all previous tokens.

    Key Innovation: We MASK the prompt tokens so the model only learns to
    generate the teacher's response, not memorize the problem statement.

    Args:
        model: Student model being trained
        tokenizer: Converts text ↔ token IDs
        problem: Math problem as text
        teacher_text: Teacher's reasoning + answer (what we want student to learn)
        device: 'cuda' or 'cpu'
        max_seq_len: Maximum sequence length to prevent GPU out-of-memory errors

    Returns:
        torch.Tensor: Loss value (lower = better mimicry)
    """
    # Format the problem into a conversational prompt
    prompt = render_prompt(problem)

    # Convert text to token IDs (numbers the model understands)
    prompt_ids = tokenizer.encode(prompt)
    teacher_ids = tokenizer.encode(teacher_text)

    # Add end-of-sequence token if not present
    if tokenizer.eos_token_id is not None and teacher_ids[-1] != tokenizer.eos_token_id:
        teacher_ids.append(tokenizer.eos_token_id)

    # Concatenate: [PROMPT tokens] + [TEACHER tokens]
    full_ids = prompt_ids + teacher_ids

    # 🔥 CRITICAL FIX: Truncate to prevent GPU memory overflow
    # Google Colab T4 GPU has limited VRAM (~15GB). Long sequences can crash training.
    if len(full_ids) > max_seq_len:
        print(f"      ⚠️  Truncating sequence: {len(full_ids)} → {max_seq_len} tokens (GPU memory protection)")
        full_ids = full_ids[:max_seq_len]

    # Convert to PyTorch tensor and add batch dimension [1, seq_len]
    input_tensor = torch.tensor(full_ids, device=device).unsqueeze(0)

    # ──────────────────────────────────────────────────────
    # FORWARD PASS: Get model predictions
    # ──────────────────────────────────────────────────────
    logits = model(input_tensor)  # Shape: [1, seq_len, vocab_size]

    # ──────────────────────────────────────────────────────
    # PREPARE FOR NEXT-TOKEN PREDICTION LOSS
    # ──────────────────────────────────────────────────────
    # In language modeling, we predict token[i+1] given tokens[0:i]
    # So we shift logits and labels by 1 position

    shift_logits = logits[0, :-1, :].contiguous()  # Predictions for positions 0 to n-1
    shift_labels = torch.tensor(full_ids[1:], device=device).contiguous()  # True tokens at positions 1 to n

    # ──────────────────────────────────────────────────────
    # MASK THE PROMPT: Don't train on problem statement
    # ──────────────────────────────────────────────────────
    # We only want the model to learn the teacher's RESPONSE, not regurgitate
    # the problem. So we set prompt tokens to -100 (ignored by loss function).

    prompt_len = min(len(prompt_ids), max_seq_len)  # Handle truncation edge case
    shift_labels[:prompt_len - 1] = -100  # Mask everything before teacher's response

    # ──────────────────────────────────────────────────────
    # COMPUTE CROSS-ENTROPY LOSS
    # ──────────────────────────────────────────────────────
    loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)  # -100 tokens are ignored
    loss = loss_fct(shift_logits, shift_labels)

    return loss


# ═══════════════════════════════════════════════════════════════════════════════
# EVALUATION: Test Student Model on Unseen Problems
# ═══════════════════════════════════════════════════════════════════════════════

import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import render_prompt, extract_final_candidate, load_math500_test
from reasoning_from_scratch.bonus.parser import normalize_text_hybrid
from reasoning_from_scratch.qwen3 import KVCache

# ---------------------------------------------------------
# STEP 3: Text Generation Function (Greedy Decoding)
# ---------------------------------------------------------
@torch.inference_mode()  # Disable gradient computation for faster inference
def generate_greedy(model, tokenizer, prompt, device, max_new_tokens=512):
    """
    Generates text one token at a time using greedy decoding (always pick most likely token).

    This is the simplest generation strategy. More advanced methods include:
    - Beam Search: Track multiple hypotheses
    - Sampling: Introduce randomness for creativity
    - Top-k/Top-p: Sample from truncated probability distribution

    Args:
        model: The student model (post-training)
        tokenizer: Text ↔ token converter
        prompt: Input question
        device: 'cuda' or 'cpu'
        max_new_tokens: Maximum response length (512 = plenty of room for reasoning)

    Returns:
        str: Generated response text
    """
    # Encode prompt and prepare for generation
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    # Key-Value Cache: Stores attention computations to avoid redundant calculation
    # This speeds up generation significantly for transformer models
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # Get initial logits (predictions for next token)
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]  # Only need last position
    generated = []

    # ──────────────────────────────────────────────────────
    # AUTOREGRESSIVE GENERATION LOOP
    # ──────────────────────────────────────────────────────
    for _ in range(max_new_tokens):
        # Pick the most likely next token (greedy = argmax)
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        # Stop if we hit the end-of-sequence token
        if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:
            break

        generated.append(next_token.item())

        # Feed the new token back into the model to predict the NEXT token
        logits = model(next_token, cache=cache)[:, -1]

    # Decode token IDs back to human-readable text
    return tokenizer.decode(generated)

# ---------------------------------------------------------
# STEP 4: Evaluation Loop (Measure Distillation Success)
# ---------------------------------------------------------
def evaluate_distilled_model(model, tokenizer, dataset_size=5):
    """
    Tests the student model on unseen math problems from MATH-500 benchmark.

    Success Metrics:
    1. Accuracy: Does it get the right answer?
    2. Style Transfer: Does it use <think> tags like the teacher?

    Args:
        model: Trained student model
        tokenizer: Text converter
        dataset_size: Number of test problems (5 for quick check, 500 for full eval)
    """
    device = get_device()
    print(f"🖥️  Using device: {device}")

    # Switch model to evaluation mode (disables dropout, etc.)
    model.eval()

    # Load benchmark test data (held-out, never seen during training)
    math_eval_data = load_math500_test()
    if dataset_size is not None:
        math_eval_data = math_eval_data[:dataset_size]

    num_correct = 0

    print(f"\n🧪 Starting Evaluation on {len(math_eval_data)} unseen questions...")
    print("👀 Watching for <think> tags to see if student learned teacher's reasoning style!\n")

    # ──────────────────────────────────────────────────────
    # EVALUATE EACH TEST PROBLEM
    # ──────────────────────────────────────────────────────
    for i, row in enumerate(math_eval_data, 1):
        prompt = render_prompt(row["problem"])

        # Generate student's answer
        answer_text = generate_greedy(model, tokenizer, prompt, device)

        # Check if student adopted teacher's formatting style
        used_think_tags = "<think>" in answer_text and "</think>" in answer_text

        # Extract the final numerical/algebraic answer from \\boxed{...}
        extracted = extract_final_candidate(answer_text)

        # ──────────────────────────────────────────────────────
        # GRADE THE ANSWER (Hybrid Parser handles edge cases)
        # ──────────────────────────────────────────────────────
        pred = normalize_text_hybrid(extracted)  # Normalize student's answer
        gold = normalize_text_hybrid(row["answer"])  # Normalize ground truth
        is_correct = (pred == gold)
        num_correct += int(is_correct)

        # ──────────────────────────────────────────────────────
        # DISPLAY RESULTS
        # ──────────────────────────────────────────────────────
        print(f"━━━ Question {i} ━━━")
        print(f"Expected : {row['answer']}")
        print(f"Extracted: {extracted!r}")
        print(f"Formatting: {'🧠 Cloned Teacher Style!' if used_think_tags else '🤖 Standard Output'}")
        print(f"Graded   : {'✅ CORRECT' if is_correct else '❌ INCORRECT'}\n")

    # ──────────────────────────────────────────────────────
    # FINAL ACCURACY REPORT
    # ──────────────────────────────────────────────────────
    accuracy = (num_correct / len(math_eval_data)) * 100
    print(f"📊 Final Distillation Accuracy: {num_correct}/{len(math_eval_data)} ({accuracy:.1f}%)")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN TRAINING LOOP: Distill Teacher Knowledge into Student
# ═══════════════════════════════════════════════════════════════════════════════

def run_ch8_distillation_training():
    """
    Complete distillation pipeline:
    1. Download pre-trained base model (student)
    2. Load teacher's outputs (training data)
    3. Train student to mimic teacher via supervised learning
    4. Evaluate on unseen problems
    """
    device = get_device()
    print(f"🖥️  Using device: {device}")

    # ──────────────────────────────────────────────────────
    # STEP 1: LOAD BASE MODEL (Student starts untrained)
    # ──────────────────────────────────────────────────────
    print("\n📥 Downloading Qwen3-0.6B base model...")
    download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")
    tokenizer = Qwen3Tokenizer(tokenizer_file_path="qwen3/tokenizer-base.json")

    # Load model in bfloat16 precision to save GPU memory
    # bfloat16 = 16-bit floating point (vs 32-bit), ~50% memory reduction
    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.to(torch.bfloat16)  # Convert before loading weights
    model.load_state_dict(torch.load("qwen3/qwen3-0.6B-base.pth", map_location=device, weights_only=True))
    model.to(device)

    # ──────────────────────────────────────────────────────
    # STEP 2: LOAD TEACHER'S TRAINING DATA
    # ──────────────────────────────────────────────────────
    distill_data = load_distillation_sample()
    STEPS = len(distill_data)

    # ──────────────────────────────────────────────────────
    # STEP 3: SETUP OPTIMIZER (AdamW is standard for LLMs)
    # ──────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)  # Learning rate: 0.00005
    model.train()  # Enable training mode (activates dropout, etc.)

    print(f"\n🎓 Starting Distillation (SFT) for {STEPS} training examples...")

    # ──────────────────────────────────────────────────────
    # STEP 4: TRAINING LOOP
    # ──────────────────────────────────────────────────────
    for step, example in enumerate(distill_data, 1):
        problem = example.get("problem", "")

        # 🔧 RECONSTRUCT TEACHER'S FULL RESPONSE
        # The data is split into "thinking" and "content" keys, we combine them
        thinking = example.get("message_thinking", "")
        content = example.get("message_content", "")
        teacher_text = f"<think>\n{thinking}\n</think>\n{content}"

        print(f"\n[Step {step}/{STEPS}] 📚 Distilling: {problem[:50]}...")

        # ──────────────────────────────────────────────────────
        # GRADIENT DESCENT STEP
        # ──────────────────────────────────────────────────────
        optimizer.zero_grad()  # Clear old gradients
        loss = compute_sft_loss(model, tokenizer, problem, teacher_text, device)
        loss.backward()  # Compute gradients via backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Prevent exploding gradients
        optimizer.step()  # Update model weights

        # Free up GPU memory (important for Colab's limited VRAM)
        torch.cuda.empty_cache()

        print(f"   📉 SFT Loss: {loss.item():.4f}")

    print("\n✅ Distillation complete! Student has cloned teacher's reasoning style.")

    # ──────────────────────────────────────────────────────
    # STEP 5: EVALUATE ON UNSEEN TEST SET
    # ──────────────────────────────────────────────────────
    try:
        evaluate_distilled_model(model, tokenizer, dataset_size=50)
    except NameError:
        print("🚨 Error: Model not in memory. Run training block first!")

# ═══════════════════════════════════════════════════════════════════════════════
# EXECUTE TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
run_ch8_distillation_training()

Using NVIDIA CUDA GPU
Using device: cuda
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
Fetching Teacher Distillation Data...

Starting Distillation (SFT) for 5 steps...

[Step 1] Distilling knowledge for: Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\tex...
      [Warning] Truncating sequence from 2927 down to 1024 tokens to prevent OOM.
   📉 SFT Loss: 0.6797

[Step 2] Distilling knowledge for: A rectangular band formation is a formation with $...
      [Warning] Truncating sequence from 6452 down to 1024 tokens to prevent OOM.
   📉 SFT Loss: 0.7227

[Step 3] Distilling knowledge for: What is the degree of the polynomial $(4 +5x^3 +10...
      [Warning] Truncating sequence from 1691 down to 1024 tokens to prevent OOM.
   📉 SFT Loss: 1.0547

[Step 4] Distilling knowledge for: Evaluate $\left\lceil3\left(6-\frac12\right)\right...
   📉 SFT Loss: 0.5586

[Step 5] Distilling knowledge for: Sam is hired for a 20-day period. On days that he ...
      [Warning] Truncating sequence from 1570 

KeyboardInterrupt: 